<a href="https://colab.research.google.com/github/drshahizan/Python-big-data/blob/main/bigData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

NURFARRAHIN CHE ALIAS (A20EC0121)
NUR IRDINA ALIAH BINTI ABDUL WAHAB (A20EC0115)

In [ ]:
! pip install kaggle



In [ ]:
from google.colab import files

files.upload()

In [ ]:
! mkdir ~/.kaggle


In [ ]:
! cp kaggle.json ~/.kaggle/

In [ ]:
! chmod 600 ~/.kaggle/kaggle.json


In [ ]:
 ! KAGGLE_CONFIG_DIR=/content/ kaggle datasets download -d elemento/nyc-yellow-taxi-trip-data

In [ ]:
!unzip nyc-yellow-taxi-trip-data.zip

In [ ]:
import pandas 
import random

filename = "/content/yellow_tripdata_2016-03.csv" 
n = sum(1 for line in open(filename))-1  # Calculate number of rows in file
s = n//10  # sample size of 10%
skip = sorted(random.sample(range(1, n+1), n-s))  # n+1 to compensate for header 
df = pandas.read_csv(filename, skiprows=skip)
df.head(5)

In [ ]:
import pandas as pd
fare_amount_max = []

for reader in pd.read_csv('/content/yellow_tripdata_2016-03.csv', chunksize=100000):
    # do any processing on reader
    fare_amount_max.append(reader['fare_amount'].max())
    

In [ ]:
max(fare_amount_max)


## Big Data Scalling

---



reduce data type


In [ ]:
#efore 
import pandas as pd
df = pd.read_csv('/content/yellow_tripdata_2016-03.csv')
df.dtypes


In [ ]:

df.memory_usage().sum()/(1024*1024*1024)

In [ ]:
import pandas as pd
import numpy as np

def reduce_mem_usage(df):
    start_mem = df.memory_usage().sum() / 1024**3
    print('Memory usage of dataframe is {:.2f} MB'.format(start_mem))
    
    for col in df.columns:
        col_type = df[col].dtype
        
        if col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)  
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
        else:
            df[col] = df[col].astype('category')

    end_mem = df.memory_usage().sum() / 1024**3
    print('Memory usage after optimization is: {:.2f} MB'.format(end_mem))
    print('Decreased by {:.1f}%'.format(100 * (start_mem - end_mem) / start_mem))
    
    return df

In [ ]:
df_new = reduce_mem_usage(df)

In [ ]:
df_new.dtypes


**Technique** #1: Don?t load all the columns

In [ ]:
>>> import pandas as pd
>>> df = pd.read_csv("yellow_tripdata_2016-03.csv")
>>> df.info(verbose=False, memory_usage="deep")

Technique #2: Shrink numerical columns with smaller dtypes

In [ ]:
 import pandas as pd
 df = pd.read_csv("yellow_tripdata_2016-03.csv")
df["fare_amount"].memory_usage(index=False, deep=True)

In [ ]:
df["fare_amount"].max()

In [ ]:
df["fare_amount"].min()

In [ ]:
df = pd.read_csv("yellow_tripdata_2016-03.csv", dtype={"fare_amount": "float16"})
df["fare_amount"].memory_usage(index=False, deep=True)

Technique #3: Shrink categorical data using Categorical dtypes


In [ ]:
df = pd.read_csv(
...     "yellow_tripdata_2016-03.csv", dtype={"VendorID": "int8"})

In [ ]:
df["VendorID"].head(5)

Technique #4: Sparse series


In [ ]:
df.memory_usage(index=False, deep=True)

## Data Cleaning

---



In [ ]:
df.head(20)

In [ ]:
df.columns

In [ ]:
print("Missing values distribution: ")
print(df.isnull().mean())
print("")

In [ ]:
print("Column datatypes: ")
print(df.dtypes)


In [ ]:
# names of the columns
columns = ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'pickup_longitude',
       'pickup_latitude', 'RatecodeID', 'store_and_fwd_flag',
       'dropoff_longitude', 'dropoff_latitude', 'payment_type', 'fare_amount',
       'extra', 'mta_tax', 'tip_amount', 'tolls_amount',
       'improvement_surcharge', 'total_amount']
   

# looping through the columns to fill the entries with NaN values with ""
for column in columns:
    df[column] = df[column].fillna("")

In [ ]:
df.head()

In [ ]:
df['tpep_pickup_datetime'].unique()

In [ ]:
df['improvement_surcharge'].unique()

In [ ]:
df = df.drop_duplicates( keep='last',inplace = True)

## Data Visualisation

---



Exploratory Analysis and Visualization
This section will shows steps and details of the calculation below:

Determining the mean, sum, range and other interesting statistics for numeric columns
Calculation distributions of numeric columns using histograms.
Explaining the relationship between columns using scatter plots and bar charts.

Graph 1 Count plot for Top 10 Car brand.

Graph 2 KDE plot to determine the average horsepower of a car.

Graph 3 Barplot and pie chart to show fueltype of car

Graph 4 Histplot for car mileage

Graph 5 Heatmap graph and Lmplot to show correlation between price and other factors

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
from wordcloud import WordCloud, STOPWORDS

In [ ]:
data = pd.read_csv( '/content/yellow_tripdata_2016-03.csv')

print(data.head())
print(data.shape)
print(data.info)

In [ ]:
sns.set(rc={'figure.figsize':(10,10)})
plt.xticks(range(0, 10, 1))
sns.kdeplot(x = 'tip_amount',data = pd.read_csv( '/content/yellow_tripdata_2016-03.csv'))
plt.title("Average Tip Amount")

In [ ]:
sns.histplot(data=pd.read_csv( '/content/yellow_tripdata_2016-03.csv'), x='passenger_count',bins=30,color = 'r')

In [ ]:
data=pd.read_csv( '/content/yellow_tripdata_2016-03.csv')

Index(['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'pickup_longitude',
       'pickup_latitude', 'RatecodeID', 'store_and_fwd_flag',
       'dropoff_longitude', 'dropoff_latitude', 'payment_type', 'fare_amount',
       'extra', 'mta_tax', 'tip_amount', 'tolls_amount',
       'improvement_surcharge', 'total_amount'],
      dtype='object')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
from wordcloud import WordCloud, STOPWORDS

In [ ]:
data=pd.read_csv( '/content/yellow_tripdata_2016-03.csv')

In [ ]:
data['payment_type'].value_counts().plot(kind='pie', figsize=(6,6))

In [ ]:
data=pd.read_csv( '/content/yellow_tripdata_2016-03.csv')
corr = data.corr()
plt.figure(figsize=(20, 9))
k = 18 #number of variables for heatmap
cols = corr.nlargest(k, 'fare_amount')['fare_amount'].index
cm = np.corrcoef(data[cols].values.T)
sns.set(font_scale=1.25)
hm = sns.heatmap(cm, cbar=True, annot=True, square=True, fmt='.2f', annot_kws={'size': 10}, yticklabels=cols.values, xticklabels=cols.values,cmap="Blues")
plt.show()

In [ ]:
sns.lmplot(data=pd.read_csv( '/content/yellow_tripdata_2016-03.csv'),x='passenger_count',y='total_amount')